# 🎯 Phase 2-2: Hansen 접근성 모델을 통한 최종 입지 선정

---
**hansen 접근법**: 단순 거리 기준은 시설까지의 물리적 근접성만 보여줄 뿐, 그 시설이 실제로 얼마나 큰 수요를 갖는지는 반영하지 못합니다.
반면 Hansen 접근법은 거리와 함께 유동인구·상주인구 등 수요 규모를 함께 고려하므로, 후보지의 실제 입지 경쟁력을 더 현실적으로 평가할 수 있습니다.

**목적**: 4개 후보지 중 접근성이 가장 높은 최종 입지 1곳을 선정  
**방법**: 수요 데이터 수집 → OSMnx 도로망 거리 → Hansen 접근성 점수 비교  
**결과**: 접근성 점수 3,715.13의 후보지 ④(남쪽) 최종 선정

**Hansen 함수**: A_i = Σ_j (P_j / dx_ij^β)  
- P_j: 수요 크기 (버스 유동인구, 상권 유동인구, 아파트 상주인구)  
- dx_ij: OSMnx 기반 도보 네트워크 거리  
- β = 1.5 (거리 감쇠 계수)

## 1. 후보지 주변 상권 데이터 확인

성현동·중앙동·은천동의 상권 영역 좌표를 확인합니다.

In [1]:
import pandas as pd

In [39]:
성현동=pd.read_csv('성현동.csv')

In [41]:
성현동

,상권_구분_코드,상권_구분_코드_명,상권_코드,상권_코드_명,엑스좌표_값,와이좌표_값,자치구_코드,자치구_코드_명,행정동_코드,행정동_코드_명,영역_면적
0,A,골목상권,3110891,구암초등학교,195414,443215,11620,관악구,11620565,성현동,40509
1,A,골목상권,3110894,관악동부센트레빌아파트(은천로35라길),195881,443185,11620,관악구,11620565,성현동,30920


In [43]:
성현동[['상권_구분_코드_명','상권_코드']]

,상권_구분_코드_명,상권_코드
0,골목상권,3110891
1,골목상권,3110894


In [23]:
성현동[['엑스좌표_값','와이좌표_값','행정동_코드_명']]

,엑스좌표_값,와이좌표_값,행정동_코드_명
0,195414,443215,성현동
1,195881,443185,성현동


In [11]:
중앙동=pd.read_csv('중앙동.csv')

In [45]:
중앙동

,상권_구분_코드,상권_구분_코드_명,상권_코드,상권_코드_명,엑스좌표_값,와이좌표_값,자치구_코드,자치구_코드_명,행정동_코드,행정동_코드_명,영역_면적
0,A,골목상권,3110892,관악구 중앙길,195677,442761,11620,관악구,11620615,중앙동,259916
1,R,전통시장,3130293,봉천중앙시장,195973,442762,11620,관악구,11620615,중앙동,17101


In [15]:
은천동=pd.read_csv('은천동.csv')

In [49]:
은천동

,상권_구분_코드,상권_구분_코드_명,상권_코드,상권_코드_명,엑스좌표_값,와이좌표_값,자치구_코드,자치구_코드_명,행정동_코드,행정동_코드_명,영역_면적
0,A,골목상권,3110884,봉천역 4번(봉천110안전센터),194626,442864,11620,관악구,11620605,은천동,43849
1,A,골목상권,3110889,봉천역 6번,195155,442768,11620,관악구,11620605,은천동,42797
2,R,전통시장,3130289,봉일시장,194553,442944,11620,관악구,11620605,은천동,4844
3,R,전통시장,3130292,봉천현대시장,195277,443011,11620,관악구,11620605,은천동,5418


## 2. 버스정거장 인근 후보지 4곳 위치 시각화 (PPT p.15)

In [1]:
import folium
from pyproj import Transformer

# 기존 데이터 (EPSG:5181)
data = {
    "성현동": [(195414, 443215), (195881, 443185)],
    "중앙동": [(195677, 442761), (195973, 442762)],
    "은천동": [(194626, 442864), (195155, 442768), (194553, 442944), (195277, 443011)]
}

# 색상 지정
color_map = {
    "성현동": "blue",
    "중앙동": "green",
    "은천동": "red"
}

# 지도 생성 (서울 중심 기준)
m = folium.Map(location=[37.48, 126.95], zoom_start=14)

# 기존 좌표 추가 (EPSG:5181 -> EPSG:4326 변환 후 표시)
transformer = Transformer.from_crs("EPSG:5181", "EPSG:4326", always_xy=True)
for dong, coords in data.items():
    color = color_map.get(dong, "gray")
    for x, y in coords:
        lon, lat = transformer.transform(x, y)
        folium.Marker(location=[lat, lon], popup=f"{dong}", icon=folium.Icon(color=color)).add_to(m)

# 구글 지도 좌표 추가 (EPSG:4326 그대로 사용)
google_coords = {
    "서": (37.489956, 126.946117),
    "북": (37.486369, 126.942261),
    "동": (37.486563, 126.951429),
    "남": (37.484903, 126.946698)
}

# 구글 지도 좌표 추가
for label, (lat, lon) in google_coords.items():
    folium.Marker(location=[lat, lon], popup=f"{label}(구글)", icon=folium.Icon(color="purple", icon="star")).add_to(m)

# 지도 저장
m


## 3. 거리감쇠계수 설정 근거

> **β = 1.5 설정 근거**:  
> - 소상공인시장진흥공단: 800~1000m를 "생활권 외곽"으로 간주  
> - 서울시 교통연구원(2023): 일상적 도보 방문의 80%는 500m 이내 발생  
> - 1km 초과 시 도보 방문 비율 15% 미만으로 감소

## 4. 상권 유동인구 집계

각 후보지에 대응하는 상권의 유동인구를 집계합니다.

In [5]:
import pandas as pd

In [35]:
상권유동인구=pd.read_csv('서울시 상권분석서비스(길단위인구-상권).csv',encoding='cp949')

In [37]:
상권유동인구
북 3110891
동 3110894
서 3110884
남 3110889	


,기준_년분기_코드,상권_구분_코드,상권_구분_코드_명,상권_코드,상권_코드_명,총_유동인구_수,남성_유동인구_수,여성_유동인구_수,연령대_10_유동인구_수,연령대_20_유동인구_수,...,시간대_14_17_유동인구_수,시간대_17_21_유동인구_수,시간대_21_24_유동인구_수,월요일_유동인구_수,화요일_유동인구_수,수요일_유동인구_수,목요일_유동인구_수,금요일_유동인구_수,토요일_유동인구_수,일요일_유동인구_수
0,20243,U,관광특구,3001496,강남 마이스 관광특구,108678,52145,56533,9633,22198,...,27529,22375,5443,15107,15971,16900,17793,17480,13833,11595
1,20243,U,관광특구,3001495,잠실 관광특구,4063096,1951882,2111214,401325,933719,...,636300,874238,481045,561847,565577,580206,585075,598772,612944,558674
2,20243,U,관광특구,3001494,종로?청계 관광특구,8089410,4277573,3811837,318059,1524498,...,1676894,1601264,648657,1260648,1264195,1275978,1300151,1321763,948075,718602
3,20243,U,관광특구,3001493,동대문패션타운 관광특구,3296772,1548190,1748582,190195,638165,...,520865,634434,391686,501400,506867,516127,529054,505175,378153,359995
4,20243,U,관광특구,3001492,명동 남대문 북창동 다동 무교동 관광특구,6988616,3346879,3641736,315668,1242415,...,1700786,1323575,349661,1113461,1134683,1138578,1158635,1159239,713044,570974
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37935,20191,A,골목상권,3110005,세검정,129171,55368,73803,19624,15305,...,13667,21367,20706,18686,18746,18727,18736,18576,17782,17918
37936,20191,A,골목상권,3110004,대신고등학교,495628,226246,269381,94335,57494,...,70258,94736,65414,69892,70158,71562,71577,70647,70556,71236
37937,20191,A,골목상권,3110003,세검정초등학교,706686,310470,396215,128431,82037,...,74478,96768,85012,100783,100429,100067,99815,98912,102760,103921
37938,20191,A,골목상권,3110002,독립문역 1번,540585,246868,293717,96885,65609,...,75590,96631,70361,76301,77283,79044,78826,78583,75041,75508


In [107]:
북=상권유동인구[상권유동인구['상권_코드']==3110891]['총_유동인구_수'].sum()
# 3110891
# 3110894
# 3110884
# 3110889	
북

4843413

In [109]:
동=상권유동인구[상권유동인구['상권_코드']==3110894]['총_유동인구_수'].sum()
동

7606376

In [111]:
서=상권유동인구[상권유동인구['상권_코드']==3110884]['총_유동인구_수'].sum()
서

17876720

In [113]:
남=상권유동인구[상권유동인구['상권_코드']==3110889]['총_유동인구_수'].sum()
남

10436363

## 5. 버스정류장 유동인구 (승하차 가중치)

> 💡 **가중치 설계 근거** (한국교통연구원, 2020)  
> 하차 인구의 구매 전환율: 25~30% → 가중치 **0.7**  
> 승차 인구의 구매 전환율: 10~15% → 가중치 **0.3**

In [77]:
# 버스정류장 유동인구

버스유동인구=pd.read_csv('BUS_STATION_BOARDING_MONTH_202501.csv',encoding='cp949')

In [95]:
# 연구 결과에 따르면, 하차 인구의 구매 전환율이 25~30% 수준.
#승차 인구의 구매 전환율은 10~15% 수준.
#한국교통연구원(2020): ‘도시형 대중교통 환승지점의 상권 특성 분석
따라서 가중치를 하차인원 0.7 승차인원 0.3에 둠
#북쪽
버스유동인구[버스유동인구['버스정류장ARS번호']=='21254']
#서쪽
21201
#동쪽
#서쪽

,사용일자,노선번호,노선명,표준버스정류장ID,버스정류장ARS번호,역명,승차총승객수,하차총승객수,등록일자
2112,20250101,5535,5535번(하안동~노량진),120000152,21254,구암초등학교(00030),59,22,20250104
3383,20250101,500,500번(석수역~을지로입구),120000152,21254,구암초등학교(00023),106,54,20250104
41021,20250102,500,500번(석수역~을지로입구),120000152,21254,구암초등학교(00023),321,144,20250105
42291,20250102,5535,5535번(하안동~노량진),120000152,21254,구암초등학교(00030),170,39,20250105
63704,20250102,8551,8551번(봉천역~노량진역),120000152,21254,구암초등학교(00004),52,1,20250105
...,...,...,...,...,...,...,...,...,...
1181223,20250130,500,500번(석수역~을지로입구),120000152,21254,구암초등학교(00023),114,50,20250202
1182488,20250130,5535,5535번(하안동~노량진),120000152,21254,구암초등학교(00030),76,31,20250202
1222086,20250131,500,500번(석수역~을지로입구),120000152,21254,구암초등학교(00023),277,143,20250203
1223364,20250131,5535,5535번(하안동~노량진),120000152,21254,구암초등학교(00030),175,31,20250203


In [103]:
버스정류장가중치=pd.read_csv('버스정류장가중치.csv')
버스정류장가중치

,Unnamed: 0,ARS번호,총 승차 인구,총 하차 인구,단순 합산(승+하),가중치 적용(0.3:0.7)
0,북쪽(21254),21254,13428,4602,18030,7249.8
1,서쪽(21201),21201,6399,12813,19212,10888.8
2,동쪽(21198),21198,5831,16908,22739,13584.9
3,남쪽(21187),21187,5815,8774,14589,7886.3


## 6. 수요 데이터 통합

아파트 상주인구: 관악구 인구 통계 기반 1세대당 1.9명 추정

In [121]:
# 최종 수요 데이터프레임 생성
final_demand_df = pd.DataFrame({
    '방향': ['북쪽', '서쪽', '동쪽', '남쪽'],
    '버스 정류장 유동 인구(가중치)': [7249.8, 10888.8, 13584.9, 7886.3],
    '상권 유동 인구': [4843413, 17876720, 7606376, 10436363],
    '아파트 상주 인구(추정)': [1894.3, 3801.9, 925.3, 1064],
})

# 총 수요 계산
final_demand_df['총 수요'] = (
    final_demand_df['버스 정류장 유동 인구(가중치)'] +
    final_demand_df['상권 유동 인구'] +
    final_demand_df['아파트 상주 인구(추정)']
)

In [123]:
final_demand_df

,방향,버스 정류장 유동 인구(가중치),상권 유동 인구,아파트 상주 인구(추정),총 수요
0,북쪽,7249.8,4843413,1894.3,4852557.1
1,서쪽,10888.8,17876720,3801.9,17891410.7
2,동쪽,13584.9,7606376,925.3,7620886.2
3,남쪽,7886.3,10436363,1064.0,10445313.3


## 7. Hansen 접근성 점수 계산

-거리는 네이버지도상에서 직접 계산하여 절삭하여 사용

In [133]:
import pandas as pd

# 사용자 제공 도로망 거리 데이터
distance_data = pd.DataFrame({
    '방향': ['북', '서', '동', '남'],
    '버스 정류장 거리(m)': [50, 50, 50, 50],
    '아파트 거리(m)': [50, 200, 50, 50],
    '상권 거리(m)': [250, 300, 400, 200],
    '버스 유동 인구(가중치)': [7249.8, 10888.8, 13584.9, 7886.3],
    '상권 유동 인구': [4843413, 17876720, 7606376, 10436363],
    '아파트 상주 인구(추정)': [1894.3, 3801.9, 925.3, 1064]
})

# Hansen 접근성 점수 계산 (β=1.5)
beta = 1.5

# Hansen 접근성 점수 계산 함수
def calculate_hansen_score(row):
    bus_score = row['버스 유동 인구(가중치)'] / (row['버스 정류장 거리(m)'] ** beta)
    market_score = row['상권 유동 인구'] / (row['상권 거리(m)'] ** beta)
    apartment_score = row['아파트 상주 인구(추정)'] / (row['아파트 거리(m)'] ** beta)
    return round(bus_score + market_score + apartment_score, 2)

# Hansen 접근성 점수 계산
distance_data['Hansen 접근성 점수'] = distance_data.apply(calculate_hansen_score, axis=1)

In [135]:
distance_data

,방향,버스 정류장 거리(m),아파트 거리(m),상권 거리(m),버스 유동 인구(가중치),상권 유동 인구,아파트 상주 인구(추정),Hansen 접근성 점수
0,북,50,50,250,7249.8,4843413,1894.3,1251.16
1,서,50,200,300,10888.8,17876720,3801.9,3472.52
2,동,50,50,400,13584.9,7606376,925.3,991.84
3,남,50,50,200,7886.3,10436363,1064.0,3715.13


## 8. 결론

| 후보지 | 버스 정류장 | 아파트 | 상권 | Hansen 점수 |
|--------|-----------|--------|------|------------|
| ① 북 | 50m | 50m | 250m | 1,251.16 |
| ② 서 | 50m | 200m | 300m | 3,472.52 |
| ③ 동 | 50m | 50m | 400m | 991.84 |
| **④ 남** | **50m** | **50m** | **200m** | **3,715.13** |

> 💡 **후보지 ④가 최종 선정된 이유**:  
남쪽은 버스정류장·아파트와의 거리가 모두 50m이고 상권과의 거리도 200m로 가장 유리해, 거리 감쇠를 반영한 Hansen 접근성 점수가 가장 높게 나타났습니다.
또한 단순히 가깝기만 한 것이 아니라 상권 유동인구 규모도 충분히 커서, 거리와 수요를 함께 고려했을 때 4개 방향 중 종합 접근성이 가장 우수한 방향으로 선택되었습니다.